In [1]:
import numpy as np
import matplotlib.pyplot as plt
import os
os.chdir('/zhome/71/c/146676/main/')
import SimpleITK as sitk
import importlib
from helpers import module_auxiliary as ma
from matplotlib_scalebar.scalebar import ScaleBar
from cil.optimisation.operators import GradientOperator # For extract edge function
import matplotlib
#matplotlib.use('Agg')
from segmentation import s3_watershed_eds
importlib.reload(s3_watershed_eds)
import matplotlib.colors as mcolors
import copy
from segmentation import s2_watershed_filter
from sklearn.neighbors import KNeighborsClassifier
from scipy.stats import mode
from scipy.spatial.distance import cosine, euclidean

In [2]:
full_seg_, segmentation_, elem_segm_, elems_ = s3_watershed_eds.get_segmentation_from_watersheds(
    thresholds = np.array([15, 15, 15, 10 , 20, 20, 30, 12, 5 , 30, 10, 13 , 92, 15, 5]),
    output=True)

elems = {}
for key in elems_:
    if key != "labels":
        
        moving = sitk.GetImageFromArray(elems_[key].astype(np.float32))
        elems[key] = s3_watershed_eds.transform_eds(elems_[key].astype(np.float32),h=2)


full_seg_, segmentation_, elem_segm_, elems_raw_ = s3_watershed_eds.get_segmentation_from_watersheds(
    thresholds = np.array([15, 15, 15, 10 , 20, 20, 30, 12, 5 , 30, 10, 13 , 92, 15, 5]),
    output=True,raw=True)

elems_raw = {}
for key in elems_:
    if key != "labels":
        
        moving = sitk.GetImageFromArray(elems_raw_[key].astype(np.float32))
        elems_raw[key] = s3_watershed_eds.transform_eds(elems_raw_[key].astype(np.float32),h=2)



In [3]:
elems_n = copy.deepcopy(elems)
elements = ['Al', 'Ca', 'Cl', 'Cr', 'Fe',  'K', 'Mg', 'Na', 'Ni',  'O',  'P',  'S', 'Si', 'Ti', 'Zr']
sum_elems = np.zeros(np.shape(elems['Al']))
for element in elements:
    sum_elems = sum_elems + elems[element]

mask = elems['O']>10
for element in elements:
    elems_n[element] = elems_n[element].astype(np.float32)
    elems_n[element][mask] = elems[element][mask]/sum_elems[mask]
    elems_n[element][np.logical_not(mask)] = 0


elems_raw_n = copy.deepcopy(elems_raw)
sum_elems_raw = np.zeros(np.shape(elems_raw['Al']))
for element in elements:
    sum_elems_raw = sum_elems_raw + elems_raw[element]

mask = elems['O']>10
for element in elements:
    elems_raw_n[element] = elems_raw_n[element].astype(np.float32)
    elems_raw_n[element][mask] = elems_raw[element][mask]/sum_elems_raw[mask]
    elems_raw_n[element][np.logical_not(mask)] = 0

/tmp/pbs.512498.hnode41/ipykernel_975747/2917447538.py:22: RuntimeWarning: invalid value encountered in divide
  elems_raw_n[element][mask] = elems_raw[element][mask]/sum_elems_raw[mask]


In [4]:
NA_surf_volume_ = np.load('/dtu-compute/msaca/cache/NA_surf1.npy')
XA_surf_volume_ = np.load('/dtu-compute/msaca/cache/XA_surf1.npy')
nz, ny, nx = np.shape(XA_surf_volume_)
subvol = [0,nz,30, 40, 0, nx]

NA_surf_volume = NA_surf_volume_[subvol[0]:subvol[1],subvol[2]:subvol[3],subvol[4]:subvol[5]]
XA_surf_volume = XA_surf_volume_[subvol[0]:subvol[1],subvol[2]:subvol[3],subvol[4]:subvol[5]]
XA, NA = s2_watershed_filter.place_medians_in_watersheds(XA_surf_volume, NA_surf_volume,
    eta_edge = 0.005,level = 0.15, conductivity = 0.5, smoothing_iter = 15, watershed_line = False)

Watershedding with level  0.15  and smoothing iter  15
Diffusion time:  65.26035714149475
Gradient time:  11.777040481567383
Watershed time:  56.111716985702515
Assignment time:  1.1870973110198975
Number of watersheds 152082


In [21]:
h = 2
##           ['Al', 'Ca', 'Cl', 'Cr', 'Fe',  'K', 'Mg', 'Na', 'Ni',  'O',  'P',  'S', 'Si', 'Ti', 'Zr']
p_80keV = np.array([0.202, 0.366, 0.269, 0.491, 0.595,  0.325, 0.195, 0.180, 0.731,  0.168,  0.232,  0.259, 0.223, 0.405, 1.72])
p_60keV = np.array([0.278, 0.658, 0.439, 0.964, 1.205,  0.568, 0.257, 2.268, 1.512,  0.190,  0.349,  0.405, 0.321, 0.766, 3.74])
p_73keV = 7/20*p_60keV + 13/20*p_80keV
density_element = [2700, 1550, 2030, 7140, 7874, 856, 1738, 968, 8908,  1495, 1823, 1960, 2330, 4507, 6511]

## 'Feld-An-Al', 'Feld-Al-Or'', 'Pyrox':,
#            'Apatite', 'Chromite', 'Ilminite', 'Pyrite',
#            'Zircon', 'Iron_oxide'}
density_mineral = [2680, 2680, 3300, 3200, 4700, 4500, 5000, 4700, 5200]
keys = list(elems_n.keys())

# Compute the weighted sum
result = sum(1e-5*coeff * elems_n[key] for coeff, key in zip(density_element, keys))

plt.figure(figsize=(10,10))
plt.imshow(result)
a = np.mean(result[0:10])
b = np.mean(result[int(h*500):int(h*600),int(h*500):int(h*600)])
plt.clim([a-(b-a)*0.3, 2*b])
plt.colorbar()
plt.savefig('/dtu-compute/msaca/cache/plot_simXA_att1.png', dpi=300)
plt.close('all')

In [22]:
##           ['Al', 'Ca', 'Cl', 'Cr', 'Fe',  'K', 'Mg', 'Na', 'Ni',  'O',  'P',  'S', 'Si', 'Ti', 'Zr']
p_80keV = np.array([0.202, 0.366, 0.269, 0.491, 0.595,  0.325, 0.195, 0.180, 0.731,  0.168,  0.232,  0.259, 0.223, 0.405, 1.72])
p_60keV = np.array([0.278, 0.658, 0.439, 0.964, 1.205,  0.568, 0.257, 2.268, 1.512,  0.190,  0.349,  0.405, 0.321, 0.766, 3.74])
p_73keV = 7/20*p_60keV + 13/20*p_80keV
density_element = np.array([2700, 1550, 2030, 7140, 7874, 856, 1738, 968, 8908,  1495, 1823, 1960, 2330, 4507, 6511])
atomic_mass = A = np.array([26.982, 40.078, 35.450, 51.996, 55.845, 39.098, 24.305, 22.990, 58.693, 16.000, 30.974, 32.065, 28.085, 47.867, 91.224])

## 'Feld-An-Al', 'Feld-Al-Or'', 'Pyrox':,
#            'Apatite', 'Chromite', 'Ilminite', 'Pyrite',
#            'Zircon', 'Iron_oxide'}
density_mineral = np.array([2680, 2680, 3300, 3200, 4700, 4500, 5000, 4700, 5200])
keys = list(elems_n.keys())

# Compute the weighted sum
result = sum(1e-5*coeff * elems_raw_n[key] for coeff, key in zip(density_element, keys))

plt.figure(figsize=(10,10))
plt.imshow(result)
a = np.mean(result[0:10])
b = np.mean(result[int(h*500):int(h*600),int(h*500):int(h*600)])
plt.clim([a-(b-a)*0.3, 2*b])
plt.colorbar()
plt.savefig('/dtu-compute/msaca/cache/plot_simrawXA_att1.png', dpi=300)
plt.close('all')

In [23]:
plt.figure(figsize=(10,10))
plt.imshow(XA[:,7])
plt.colorbar()
a = np.mean(XA[0:10,7])
b = np.mean(XA[500:600,7,500:600])
plt.clim([a-(b-a)*0.3, 2*b])
plt.savefig('/dtu-compute/msaca/cache/plot_XA_att1.png', dpi=300)
plt.close('all')

In [27]:
# Neutron database: https://nds.iaea.org/ngatlas2/
# https://ncnr.nist.gov/resources/n-lengths/

cross_section_micro_elements = [0.231, 0.43, 33.5, 3.05, 2.56,  2.1, 0.063, 0.53, 4.49, 0.00019,  0.172,  0.53, 0.171, 6.09, 0.185] # At 2200m/s energy (around 25meV)
neutron_absorption = cross_section_micro_elements * density_element/atomic_mass

result = sum(coeff * elems_raw_n[key] for coeff, key in zip(neutron_absorption, keys))

plt.figure(figsize=(10,10))
plt.imshow(result)
a = np.mean(result[0:10])
b = np.mean(result[int(h*500):int(h*600),int(h*500):int(h*600)])
plt.clim([a-(b-a)*0.3, 2*b])
plt.colorbar()
plt.savefig('/dtu-compute/msaca/cache/plot_6_simrawNA_att.png', dpi=300)
plt.close('all')

In [25]:
# Neutron database: https://nds.iaea.org/ngatlas2/
# https://ncnr.nist.gov/resources/n-lengths/

cross_section_micro_elements = [0.231, 0.43, 33.5, 3.05, 2.56,  2.1, 0.063, 0.53, 4.49, 0.00019,  0.172,  0.53, 0.171, 6.09, 0.185] # At 2200m/s energy (around 25meV)
neutron_absorption = cross_section_micro_elements# * density_element/atomic_mass

result = sum(coeff * elems_n[key] for coeff, key in zip(neutron_absorption, keys))

plt.figure(figsize=(10,10))
plt.imshow(result)
a = np.mean(result[0:10])
b = np.mean(result[int(h*500):int(h*600),int(h*500):int(h*600)])
plt.clim([a-(b-a)*0.3, 2*b])
plt.colorbar()
plt.savefig('/dtu-compute/msaca/cache/plot_sim2NA_att.png', dpi=300)
plt.close('all')

In [ ]:
plt.figure(figsize=(10,10))
plt.imshow(XA_surf_volume[:,7])
a = np.mean(XA_surf_volume[0:10,7])
b = np.mean(XA_surf_volume[500:600,7,500:600])
plt.clim([a-(b-a)*0.3, 2*b])
plt.colorbar()
plt.savefig('/dtu-compute/msaca/cache/plot_XA_att1.png', dpi=300)
plt.close('all')